<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_DeepLearning/student/Tutorial3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 3: Depth and Representation

**Session 1: Deep Learning Foundations**

**Objective:** Understand why stacking layers matters.


## Tutorial Objectives

A multi-layer perceptron repeatedly applies linear maps and nonlinearities:

$$
\mathbf{h}_1 = f(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1), \quad
\mathbf{h}_2 = f(\mathbf{W}_2\mathbf{h}_1 + \mathbf{b}_2), \quad \dots
$$

Depth can help a network build hierarchical representations, but it can also make optimization harder.


In [ ]:
# Imports and shared settings
import random
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
def get_mnist_loaders(batch_size=128, train_subset=12000, val_size=2000):
    """Return small MNIST train/validation/test loaders for quick tutorials."""
    transform = transforms.ToTensor()

    full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    if train_subset is not None:
        indices = torch.randperm(len(full_train))[:train_subset + val_size]
        full_train = Subset(full_train, indices)

    train_size = len(full_train) - val_size
    train_data, val_data = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_mnist_loaders()
images, labels = next(iter(train_loader))
print('Image batch:', images.shape)
print('Label batch:', labels.shape)


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def fit(model, train_loader, val_loader, n_epochs=5, lr=1e-2, momentum=0.0):
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'Epoch {epoch + 1:02d} | '
            f'train loss {train_loss:.3f}, acc {train_acc:.3f} | '
            f'val loss {val_loss:.3f}, acc {val_acc:.3f}'
        )

    return history


def plot_history(history, title='Training curves'):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))

    axs[0].plot(epochs, history['train_loss'], marker='o', label='train')
    axs[0].plot(epochs, history['val_loss'], marker='o', label='validation')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title('Loss')
    axs[0].legend()

    axs[1].plot(epochs, history['train_acc'], marker='o', label='train')
    axs[1].plot(epochs, history['val_acc'], marker='o', label='validation')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_ylim(0, 1)
    axs[1].set_title('Accuracy')
    axs[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Build MLPs with Different Depths

Here, depth means the number of hidden layers. A model with depth 0 is just logistic regression.


In [ ]:
class MLP(nn.Module):
    def __init__(self, depth=1, hidden_units=128):
        super().__init__()
        layers = [nn.Flatten()]

        if depth == 0:
            layers.append(nn.Linear(28 * 28, 10))
        else:
            layers.append(nn.Linear(28 * 28, hidden_units))
            layers.append(nn.ReLU())
            for _ in range(depth - 1):
                layers.append(nn.Linear(hidden_units, hidden_units))
                layers.append(nn.ReLU())
            layers.append(nn.Linear(hidden_units, 10))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for depth in [0, 1, 2, 4]:
    model = MLP(depth=depth)
    print(f'depth={depth}: {count_parameters(model):,} trainable parameters')


## Exercise: Compare Accuracy Across Depths

Train networks with different depths using the same data and optimizer settings. Keep the experiment small so it runs quickly in Colab.


In [ ]:
depths = [0, 1, 2, 4]
depth_histories = {}
trained_models = {}

for depth in depths:
    print(f'\nTraining depth={depth}')
    torch.manual_seed(SEED)
    model = MLP(depth=depth, hidden_units=128)
    depth_histories[depth] = fit(model, train_loader, val_loader, n_epochs=5, lr=0.1)
    trained_models[depth] = model


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for depth, hist in depth_histories.items():
    epochs = np.arange(1, len(hist['train_loss']) + 1)
    axs[0].plot(epochs, hist['val_loss'], marker='o', label=f'depth={depth}')
    axs[1].plot(epochs, hist['val_acc'], marker='o', label=f'depth={depth}')

axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Validation loss')
axs[0].set_title('Validation loss by depth')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Validation accuracy')
axs[1].set_ylim(0, 1)
axs[1].set_title('Validation accuracy by depth')
for ax in axs:
    ax.legend()
plt.tight_layout()
plt.show()


## Representation Learning

A hidden layer transforms the input into a new feature space. We can visualize the first two principal components of hidden activations to see whether digits become more separated.


In [ ]:
from sklearn.decomposition import PCA

class FeatureExtractor(nn.Module):
    def __init__(self, model, stop_after_layer):
        super().__init__()
        self.features = nn.Sequential(*list(model.net.children())[:stop_after_layer])

    def forward(self, x):
        return self.features(x)

# Use the first hidden representation from the depth=2 model.
model = trained_models[2].to(DEVICE).eval()
extractor = FeatureExtractor(model, stop_after_layer=3).to(DEVICE).eval()

all_features = []
all_labels = []
with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader):
        features = extractor(images.to(DEVICE)).cpu()
        all_features.append(features)
        all_labels.append(labels)
        if batch_idx >= 5:
            break

features = torch.cat(all_features).numpy()
feature_labels = torch.cat(all_labels).numpy()
features_2d = PCA(n_components=2, random_state=SEED).fit_transform(features)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=feature_labels, cmap='tab10', s=10)
plt.colorbar(scatter, ticks=range(10), label='Digit')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('PCA of hidden representations')
plt.show()


## Discussion Questions

1. Does deeper always mean better in this experiment?
2. How does parameter count change with depth?
3. What is a feature hierarchy in the context of digit recognition?
4. What signs of optimization difficulty or diminishing returns do you observe?
